# Customer Churn Prediction - Exploratory Data Analysis (EDA)

This notebook performs a comprehensive exploratory data analysis of the Telco Customer Churn dataset.

**Objectives:**
- Load and understand the dataset
- Analyze data types and missing values
- Explore numerical and categorical features
- Analyze churn distribution
- Identify key relationships with churn
- Generate insights for feature engineering

## 1. Import Libraries

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_telco_data, clean_data, validate_data, get_dataset_info

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Libraries imported successfully")

## 2. Load and Explore the Dataset

In [ ]:
# Load dataset
df = load_telco_data('../data/WA_Fn-UseC_-_Telco_Customer_Churn.csv')

# Clean data
df = clean_data(df)

# Validate data
validate_data(df)

print(f"\nDataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Get dataset information
info = get_dataset_info(df)

print("\n=== DATASET INFORMATION ===")
print(f"Shape: {info['shape']}")
print(f"Missing values: {info['missing_values']}")
print(f"Duplicate rows: {info['duplicates']}")
print(f"Churn rate: {info['churn_rate']:.2%}")

print("\n=== DATA TYPES ===")
print(df.dtypes)

In [ ]:
# Detailed statistics
print("\n=== NUMERICAL FEATURES STATISTICS ===")
df.describe()

## 3. Churn Analysis

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

print("=== CHURN DISTRIBUTION ===")
for value in ['No', 'Yes']:
    count = churn_counts.get(value, 0)
    pct = churn_percentages.get(value, 0)
    print(f"{value}: {count} customers ({pct:.2f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(churn_counts.index, churn_counts.values, color=colors)
axes[1].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Customers')
axes[1].set_xlabel('Churn Status')

plt.tight_layout()
plt.savefig('../screenshots/01_churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved")

## 4. Numerical Features Analysis

In [ ]:
# Identify numerical columns
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numerical columns: {numerical_cols}")

# Tenure distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tenure distribution
axes[0, 0].hist(df['tenure'], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Tenure Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Tenure (months)')
axes[0, 0].set_ylabel('Frequency')

# Monthly charges distribution
axes[0, 1].hist(df['MonthlyCharges'], bins=30, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Monthly Charges Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Monthly Charges ($)')
axes[0, 1].set_ylabel('Frequency')

# Total charges distribution
axes[1, 0].hist(df['TotalCharges'], bins=30, color='#e67e22', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Total Charges Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Total Charges ($)')
axes[1, 0].set_ylabel('Frequency')

# Senior citizen distribution
if 'SeniorCitizen' in df.columns:
    senior_counts = df['SeniorCitizen'].value_counts()
    axes[1, 1].bar(['Non-Senior', 'Senior'], senior_counts.values, color=['#1abc9c', '#e74c3c'])
    axes[1, 1].set_title('Senior Citizen Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../screenshots/02_numerical_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Numerical features visualization saved")

## 5. Categorical Features Analysis

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Display value counts for each categorical feature
for col in categorical_cols[:6]:  # Show first 6
    print(f"\n{col}:")
    print(df[col].value_counts())
    print("-" * 40)

In [ ]:
# Visualize categorical features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

categorical_to_plot = ['gender', 'Partner', 'Dependents', 'PhoneService', 
                       'InternetService', 'Contract']

for idx, col in enumerate(categorical_to_plot):
    if col in df.columns:
        value_counts = df[col].value_counts()
        axes[idx].bar(range(len(value_counts)), value_counts.values, color='#3498db', alpha=0.7)
        axes[idx].set_xticks(range(len(value_counts)))
        axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
        axes[idx].set_title(f'{col}', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../screenshots/03_categorical_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Categorical features visualization saved")

## 6. Churn vs Features Relationships

In [ ]:
# Churn by tenure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Tenure vs Churn
axes[0, 0].hist([df[df['Churn'] == 'No']['tenure'], df[df['Churn'] == 'Yes']['tenure']], 
                 label=['No Churn', 'Churn'], color=['#2ecc71', '#e74c3c'], bins=30, alpha=0.7)
axes[0, 0].set_title('Tenure vs Churn', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Tenure (months)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Monthly charges vs Churn
axes[0, 1].hist([df[df['Churn'] == 'No']['MonthlyCharges'], 
                  df[df['Churn'] == 'Yes']['MonthlyCharges']], 
                 label=['No Churn', 'Churn'], color=['#2ecc71', '#e74c3c'], bins=30, alpha=0.7)
axes[0, 1].set_title('Monthly Charges vs Churn', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Monthly Charges ($)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Contract type vs Churn
if 'Contract' in df.columns:
    contract_churn = pd.crosstab(df['Contract'], df['Churn'])
    contract_churn.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'], alpha=0.7)
    axes[1, 0].set_title('Contract Type vs Churn', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')
    axes[1, 0].legend(title='Churn')

# Internet service vs Churn
if 'InternetService' in df.columns:
    internet_churn = pd.crosstab(df['InternetService'], df['Churn'])
    internet_churn.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], alpha=0.7)
    axes[1, 1].set_title('Internet Service vs Churn', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')
    axes[1, 1].legend(title='Churn')

plt.tight_layout()
plt.savefig('../screenshots/04_churn_relationships.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Churn relationships visualization saved")

## 7. Key Insights

In [ ]:
print("\n" + "="*60)
print("KEY INSIGHTS FROM EDA")
print("="*60)

# Insight 1: Churn rate
churn_rate = (df['Churn'] == 'Yes').sum() / len(df)
print(f"\n1. CHURN RATE: {churn_rate:.2%} of customers churned")
print(f"   - Total customers: {len(df):,}")
print(f"   - Churned: {(df['Churn'] == 'Yes').sum():,}")
print(f"   - Retained: {(df['Churn'] == 'No').sum():,}")

# Insight 2: Tenure effect
avg_tenure_churned = df[df['Churn'] == 'Yes']['tenure'].mean()
avg_tenure_retained = df[df['Churn'] == 'No']['tenure'].mean()
print(f"\n2. TENURE EFFECT:")
print(f"   - Avg tenure (churned): {avg_tenure_churned:.1f} months")
print(f"   - Avg tenure (retained): {avg_tenure_retained:.1f} months")

# Insight 3: Contract impact
if 'Contract' in df.columns:
    print(f"\n3. CONTRACT IMPACT:")
    for contract_type in df['Contract'].unique():
        churn_pct = (df[df['Contract'] == contract_type]['Churn'] == 'Yes').sum() / len(df[df['Contract'] == contract_type])
        print(f"   - {contract_type}: {churn_pct:.2%} churn rate")

# Insight 4: Internet service impact
if 'InternetService' in df.columns:
    print(f"\n4. INTERNET SERVICE IMPACT:")
    for service in df['InternetService'].unique():
        churn_pct = (df[df['InternetService'] == service]['Churn'] == 'Yes').sum() / len(df[df['InternetService'] == service])
        print(f"   - {service}: {churn_pct:.2%} churn rate")

# Insight 5: Monthly charges
avg_charges_churned = df[df['Churn'] == 'Yes']['MonthlyCharges'].mean()
avg_charges_retained = df[df['Churn'] == 'No']['MonthlyCharges'].mean()
print(f"\n5. MONTHLY CHARGES EFFECT:")
print(f"   - Avg charges (churned): ${avg_charges_churned:.2f}")
print(f"   - Avg charges (retained): ${avg_charges_retained:.2f}")

print("\n" + "="*60)

## 8. Summary Statistics

In [ ]:
# Summary by churn status
print("\nSUMMARY STATISTICS BY CHURN STATUS:")
print("\n=== NUMERICAL FEATURES ===")
summary = df.groupby('Churn')[['tenure', 'MonthlyCharges', 'TotalCharges']].agg(['mean', 'median', 'std'])
print(summary.round(2))

print("\n=== CATEGORICAL FEATURES ===")
for col in ['Partner', 'Dependents', 'PhoneService', 'InternetService']:
    if col in df.columns:
        print(f"\n{col}:")
        crosstab = pd.crosstab(df[col], df['Churn'], margins=True)
        print(crosstab)